# Analisis de Free-Living con metadatos clinicos (UPDRS / Hoehn & Yahr)
Procesa el fichero `parkinson_trunk_cwa_matches.csv`: descarga cada grabacion de tronco (`device_position == "Back"`), corre el analisis de marcha, y junta el resultado con la severidad clinica (UPDRS, HY) de cada paciente.

**Decisiones aplicadas:**
- Se confia en la columna `device_position` (no se excluyen los 6 casos donde el nombre del fichero no coincide).
- Pacientes con varias sesiones de tronco se **promedian** en una sola fila final.
- El identificador de paciente usado en todo el analisis es `patient_key` -- `patient_name` se descarta desde el principio (no se guarda en ningun CSV de salida).

Es ~8.5 GB en total -- el script descarga y borra cada `.cwa` uno a uno, y puede pausarse/reanudarse (no reprocesa sesiones ya completadas).

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!pip install -q scikit-digital-health requests


In [ ]:
import os
import sys
import pandas as pd

sys.path.append('/content/drive/MyDrive/Lasige')  # carpeta donde esta gait_lib.py
from gait_lib import build_gait_old, run_gait_freeliving, download_file


## 2. Configuracion

In [ ]:
MATCHES_CSV = "/content/drive/MyDrive/Lasige/parkinson_trunk_cwa_matches.csv"

DOWNLOAD_DIR = "/content/freeliving_cwa"       # disco local de Colab, NO Drive (mas rapido, se borra al terminar)
OUTPUT_DIR   = "/content/drive/MyDrive/Lasige/FreeLiving_Matched"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

PER_SESSION_CSV = os.path.join(OUTPUT_DIR, "Per_session_freeliving_gait.csv")
PER_PATIENT_CSV = os.path.join(OUTPUT_DIR, "Per_patient_freeliving_gait.csv")

MAX_SESSIONS = 5          # limite para pruebas; sube el numero cuando confirmes que funciona
DELETE_AFTER_PROCESSING = True   # borra el .cwa tras analizarlo (recomendado, son ~150-270MB cada uno)
PATIENT_HEIGHT_DEFAULT_M = 1.69  # se usa si 'height' del fichero no es un numero valido


## 3. Cargar y limpiar el manifiesto

Filtra solo `device_position == "Back"`, descarta `patient_name` de inmediato (no queremos nombres reales en ningun CSV de salida).

In [ ]:
manifest = pd.read_csv(MATCHES_CSV)

trunk_manifest = manifest[manifest["device_position"] == "Back"].copy()
trunk_manifest = trunk_manifest.drop(columns=["patient_name"], errors="ignore")

print(f"Sesiones de tronco (Back): {len(trunk_manifest)}")
print(f"Pacientes unicos: {trunk_manifest['patient_key'].nunique()}")

trunk_manifest[["patient_key", "patient_code", "filename", "mdsupdrs_total", "hoehn_and_yahr"]].head(10)


## 4. Descargar + analizar cada sesion (resumible)

Si `Per_session_freeliving_gait.csv` ya existe, las sesiones que ya aparecen ahi se saltan -- puedes parar la celda y volver a correrla despues sin perder progreso.

In [ ]:
gait = build_gait_old()

if os.path.exists(PER_SESSION_CSV):
    session_results = pd.read_csv(PER_SESSION_CSV)
    already_done = set(session_results["filename"])
    print(f"Retomando: {len(already_done)} sesiones ya procesadas anteriormente")
else:
    session_results = pd.DataFrame()
    already_done = set()

new_rows = []
processed_now = 0

for _, row in trunk_manifest.iterrows():
    if processed_now >= MAX_SESSIONS:
        break
    if row["filename"] in already_done:
        continue

    print(f"\n[{row['patient_key']} / {row['filename']}] descargando ({row['file_size_mb']:.0f} MB)...")
    local_path = os.path.join(DOWNLOAD_DIR, row["filename"])

    try:
        download_file(row["signed_download_url"], local_path)

        height_m = row["height"] / 100 if pd.notna(row["height"]) and row["height"] < 3 else (
            row["height"] / 100 if pd.notna(row["height"]) else PATIENT_HEIGHT_DEFAULT_M
        )

        summary_df = run_gait_freeliving(
            gait_model=gait,
            patient_id=row["patient_key"],
            file_path=local_path,
            output_dir=OUTPUT_DIR,
            group="PD",
            height_m=height_m,
            fs=None,
        )

        if summary_df is not None:
            summary_df["filename"] = row["filename"]
            summary_df["patient_code"] = row["patient_code"]
            summary_df["mdsupdrs_total"] = row["mdsupdrs_total"]
            summary_df["mdsupdrs_part_III"] = row["mdsupdrs_part_III"]
            summary_df["hoehn_and_yahr"] = row["hoehn_and_yahr"]
            new_rows.append(summary_df)
            processed_now += 1

    except Exception as e:
        print(f"[{row['filename']}] ERROR: {e}")

    finally:
        if DELETE_AFTER_PROCESSING and os.path.exists(local_path):
            os.remove(local_path)

if new_rows:
    session_results = pd.concat([session_results] + new_rows, ignore_index=True)
    session_results.to_csv(PER_SESSION_CSV, index=False)
    print(f"\nGuardado -> {PER_SESSION_CSV} ({len(session_results)} sesiones en total)")
else:
    print("\nNo se proceso ninguna sesion nueva (o ya estaban todas hechas).")


Vuelve a correr la Celda 4 las veces que haga falta (subiendo `MAX_SESSIONS` o dejandolo fijo) hasta que ya no queden sesiones pendientes.

## 5. Validar sesiones antes de promediar

Dos chequeos:
1. **Duracion imposible** (<=0h o >500h) -- timestamps corruptos dentro del `.cwa` (ej. un reset de reloj a mitad de grabacion). Estas sesiones se **excluyen** del promedio por paciente; si el paciente tiene otra sesion valida, se queda en el analisis solo con esa.
2. **Duracion sospechosamente corta** (< `MIN_RECORDING_HOURS`) -- no esta rota, pero unas pocas horas no son comparables con varios dias de free-living del resto del grupo. Estas **no se excluyen automaticamente** (podria ser una grabacion corta pero real) -- se marcan con `short_recording_flag=True` para que decidas tu si las quitas antes del analisis final.

In [ ]:
MIN_RECORDING_HOURS = 24  # umbral para marcar (no excluir) sesiones como "cortas"

# 1) duracion imposible -> se excluye del promedio
invalid_duration = session_results[
    (session_results["recording_hours"] <= 0) | (session_results["recording_hours"] > 500)
]
if len(invalid_duration) > 0:
    print(f"[EXCLUIDAS] {len(invalid_duration)} sesion(es) con duracion invalida:")
    print(invalid_duration[["patient_id", "filename", "recording_hours"]].to_string(index=False))

session_results = session_results[
    (session_results["recording_hours"] > 0) & (session_results["recording_hours"] <= 500)
].reset_index(drop=True)

# 2) duracion corta -> se marca, no se excluye
session_results["short_recording_flag"] = session_results["recording_hours"] < MIN_RECORDING_HOURS

short_sessions = session_results[session_results["short_recording_flag"]]
if len(short_sessions) > 0:
    print(f"\n[AVISO] {len(short_sessions)} sesion(es) con menos de {MIN_RECORDING_HOURS}h "
          f"(marcadas, no excluidas):")
    print(short_sessions[["patient_id", "filename", "recording_hours", "n_steps"]].to_string(index=False))

session_results.to_csv(PER_SESSION_CSV, index=False)


## 6. Promediar sesiones por paciente

Un paciente con varias sesiones (baseline/discharge) queda en **una sola fila**, promediando las columnas numericas. `n_sessions` indica cuantas se promediaron. `short_recording_flag` tambien se promedia -- si sale > 0, al menos una de las sesiones de ese paciente era corta.

In [ ]:
numeric_cols = session_results.select_dtypes(include="number").columns.tolist()

per_patient = (
    session_results
    .groupby("patient_id")[numeric_cols]
    .mean()
    .reset_index()
    .rename(columns={"patient_id": "patient_key"})
)

n_sessions = session_results.groupby("patient_id").size().rename("n_sessions")
per_patient = per_patient.merge(n_sessions, left_on="patient_key", right_index=True)

# patient_code no es numerico -- tomamos el primero (es el mismo paciente en todas las filas)
codes = session_results.groupby("patient_id")["patient_code"].first()
per_patient = per_patient.merge(codes, left_on="patient_key", right_index=True)

per_patient.to_csv(PER_PATIENT_CSV, index=False)
print(f"Guardado -> {PER_PATIENT_CSV} ({len(per_patient)} pacientes)")

per_patient


## 7. Combinar el desglose por duracion de bout (todos los pacientes)

`run_gait_freeliving` ya guarda, por cada sesion, un CSV con las metricas desglosadas en categorias `5-15s / 15-30s / 30-60s / >60s`. Esta celda los junta todos en un unico CSV, para analizar patrones de bouts cortos vs. largos en todo el grupo.

In [ ]:
import glob

bout_cat_files = glob.glob(os.path.join(OUTPUT_DIR, "*_freeliving_bouts_by_duration.csv"))

if bout_cat_files:
    all_bout_categories = pd.concat(
        [pd.read_csv(f) for f in bout_cat_files], ignore_index=True
    )
    all_bout_categories_path = os.path.join(OUTPUT_DIR, "ALL_patients_freeliving_bouts_by_duration.csv")
    all_bout_categories.to_csv(all_bout_categories_path, index=False)
    print(f"Guardado -> {all_bout_categories_path} ({len(bout_cat_files)} sesiones combinadas)")
else:
    print("Aun no hay ningun *_freeliving_bouts_by_duration.csv -- corre primero la Celda 4.")

all_bout_categories.head(10) if bout_cat_files else None


## 8. Correlacion con severidad clinica

Ahora que cada paciente tiene su UPDRS/HY, podemos ver si alguna metrica de marcha en free-living se relaciona con la severidad motora.

In [ ]:
import matplotlib.pyplot as plt

X_METRIC = "mdsupdrs_part_III"   # subescala motora del UPDRS
Y_METRIC = "PARAM:cadence_mean"  # cambia por steps_per_day, PARAM:gait speed_mean, etc.

valid = per_patient[[X_METRIC, Y_METRIC]].dropna()
r = valid.corr().iloc[0, 1] if len(valid) > 2 else float("nan")

plt.figure(figsize=(6, 5))
plt.scatter(valid[X_METRIC], valid[Y_METRIC], alpha=0.8)
plt.xlabel(X_METRIC)
plt.ylabel(Y_METRIC)
plt.title(f"r = {r:.2f}  (n={len(valid)})")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "FreeLiving_severity_correlation.png"), dpi=200)
plt.show()
